<a href="https://colab.research.google.com/github/KauePetrica/Consumindo-APIs/blob/Apis/API_PINTEREST_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests
import pandas as pd
from google.cloud import storage, bigquery
from google.api_core.exceptions import NotFound
import re

def sanitize_column_name(name):
    """
    Substitui caracteres especiais e espaços por underscores e remove caracteres inválidos.
    """
    return re.sub(r'[^a-zA-Z0-9_]', '_', name)

def upload_to_gcs(local_filename, bucket_name, destination_blob_name):
    """
    Faz upload de um arquivo para o Google Cloud Storage.
    """
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(local_filename)
    print(f"Arquivo enviado para o GCS: {destination_blob_name} em {bucket_name}")

def upload_to_bigquery(df, project_id, dataset_id, table_id):
    """
    Carrega um DataFrame para o BigQuery, criando a tabela se necessário e evitando duplicação.
    """
    client_bq = bigquery.Client(project=project_id)
    table = f"{dataset_id}.{table_id}"

    # Definir o esquema da tabela com base no cabeçalho do DataFrame
    schema = []
    for column in df.columns:
        if df[column].dtype == 'object':
            field_type = 'STRING'
        elif df[column].dtype == 'float64':
            field_type = 'FLOAT'
        elif df[column].dtype == 'int64':
            field_type = 'INT64'
        elif df[column].dtype == 'datetime64[ns]':
            field_type = 'DATETIME'
        else:
            field_type = 'STRING'
        schema.append(bigquery.SchemaField(column, field_type))
    job_config = bigquery.LoadJobConfig(schema=schema)

    # Verificar se a tabela existe e criar se não existir
    try:
        client_bq.get_table(table)
        print(f"Tabela {table} já existe.")
    except NotFound:
        table_ref = client_bq.dataset(dataset_id).table(table_id)
        table = bigquery.Table(table_ref, schema=schema)
        client_bq.create_table(table)
        print(f"Tabela {table} criada com sucesso.")

    # Inserir novos dados evitando duplicação
    temp_table = f"{dataset_id}.TEMP_{table_id}"
    client_bq.delete_table(temp_table, not_found_ok=True)  # Limpar tabela temporária, se existir
    load_job = client_bq.load_table_from_dataframe(df, temp_table, job_config=job_config)
    load_job.result()  # Aguarda a conclusão do job

    merge_query = f"""
    MERGE `{project_id}.{dataset_id}.{table_id}` T
    USING `{project_id}.{dataset_id}.TEMP_{table_id}` S
    ON T.Date = S.Date AND T.Campaign_ID = S.Campaign_ID
    WHEN NOT MATCHED THEN
    INSERT ROW
    """
    query_job = client_bq.query(merge_query)
    query_job.result()  # Aguarda a conclusão do job
    client_bq.delete_table(temp_table)  # Limpar tabela temporária
    print(f"Novos dados inseridos com sucesso na tabela {table}.")

def main(self):
    access_token = os.environ['access_token']
    ad_account_id = os.environ['ad_account_id']
    url = f'https://api.pinterest.com/v5/ad_accounts/{ad_account_id}/analytics'

    # Chamada de API
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    params = {
        'granularity': 'DAY',
        'columns': 'TOTAL_CLICKTHROUGH, TOTAL_CONVERSIONS',
        'start_date': '2024-04-01',
        'end_date': '2024-04-30'
    }
    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        data = response.json()
        print("Dados recebidos:", data)
    else:
        print("Erro ao fazer a solicitação:")
        print(f"Status Code: {response.status_code}")
        print(f"Response: {response.text}")
        return

    # Criar DataFrame diretamente a partir dos dados recebidos
    df = pd.json_normalize(data)
    df.columns = [sanitize_column_name(col) for col in df.columns]

    print("DataFrame criado com sucesso:")
    print(df.head())

    # Salvar o DataFrame localmente como CSV (opcional, para backup ou verificação)
    local_filename = "/tmp/report_pinterest.csv"
    df.to_csv(local_filename, index=False)

    bucket_name = 'XXXXXXX'
    destination_blob_name = 'XXX/XXXX/XXXXX.csv'
    upload_to_gcs(local_filename, bucket_name, destination_blob_name)

    project_id = 'XXXXX'
    dataset_id = 'XXXX'
    table_id = 'XXXXX'
    upload_to_bigquery(df, project_id, dataset_id, table_id)

    return "Função finalizada com sucesso!"
